In [1]:
import torch

In [5]:
torch.zeros(4, 8, 16).dim()

3

In [95]:
import math

import torch
from einops import einsum


class Linear(torch.nn.Module):
    weights: torch.Tensor

    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.weights = torch.nn.parameter.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype))
        std = math.sqrt(2 / (in_features + out_features))
        torch.nn.init.trunc_normal_(tensor=self.weights, mean=0, std=std, a=-3 * std, b=3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einsum(x, self.weights, "... in, out in -> ... out")

In [96]:
linear = Linear(1024, 1024)
linear.forward(torch.randn(1024, 1024))

list(linear.state_dict().keys())

['weights']

In [97]:
import torch
import math
from einops import einsum

class Embedding(torch.nn.Module):

    embeddings: torch.Tensor # (num_embeddings, embedding_dim)

    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.embeddings = torch.nn.parameter.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))
        torch.nn.init.trunc_normal_(tensor=self.embeddings, mean=0, std=1, a=-3, b=3)

    # token_ids: torch.LongTensor (batch_size, sequence_length)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[token_ids]

In [98]:
## embedding = Embedding(256, 1024)
embedding.forward(torch.randint(0, 256, (2, 5)))

tensor([[[ 0.2751, -1.9522, -0.7441,  ...,  0.6808,  0.3190,  0.1420],
         [ 0.0813,  1.4430, -1.3011,  ...,  1.9318,  0.5945, -0.7616],
         [ 0.6356, -0.7980, -0.2201,  ...,  0.9425, -0.3625,  0.6623],
         [-0.3024, -0.8646, -1.3335,  ..., -0.2152, -0.1568,  0.0220],
         [ 1.4593,  0.8459,  0.8911,  ..., -0.1295, -0.9997,  1.0710]],

        [[ 0.6392,  0.3861,  0.8381,  ...,  0.2904, -0.0357,  0.6120],
         [-1.0818,  1.7695,  0.2275,  ..., -1.4809, -0.8686,  0.0203],
         [-0.7884, -0.3207,  0.5159,  ...,  1.5542,  1.2491,  0.4872],
         [ 0.1313, -0.5577, -0.4686,  ..., -1.9838,  1.7162,  0.0154],
         [-0.0755, -1.0351,  0.0029,  ...,  1.2122, -0.9954,  0.6905]]],
       grad_fn=<IndexBackward0>)

In [99]:
import torch

class RMSNorm(torch.nn.Module):

    gain: torch.Tensor # (d_model, )
    eps: float
    
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.gain = torch.nn.parameter.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.eps = eps

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        x_sqrd_mean = x.pow(2).mean(dim=-1, keepdim=True) # (batch_size, sequence_length, 1)
        rms = torch.sqrt(x_sqrd_mean + self.eps) # (batch_size, sequence_length, 1)
        # (batch_size, sequence_length, d_model) * (d_model, ) / (batch_size, sequence_length, 1)
        result = x * self.gain / rms # (batch_size, sequence_length, d_model)
        return result.to(in_dtype)


In [114]:
rms_norm = RMSNorm(1024)
rms_norm.forward(torch.randn(1024))

tensor([-0.8464, -0.1628,  0.3576,  ..., -0.1147, -0.3050,  1.0986],
       grad_fn=<DivBackward0>)

In [126]:
import torch

class SwiGLU(torch.nn.Module):
    w_1: Linear # (d_ff, d_model)
    w_2: Linear # (d_model, d_ff)
    w_3: Linear # (d_ff, d_model)

    def __init__(self, d_model: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        d_ff = round(8 * d_model / 3 / 64) * 64
        self.w_1 = Linear(d_model, d_ff, device, dtype)
        self.w_2 = Linear(d_ff, d_model, device, dtype)
        self.w_3 = Linear(d_model, d_ff, device, dtype)

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_2(self._silu(self.w_1(x)) * self.w_3(x))

    def _silu(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)

In [127]:
d_model: int = 1024
swiglu = SwiGLU(d_model)
swiglu.forward(torch.randn(256, d_model))

tensor([[ 1.0154,  0.0667, -0.1968,  ...,  0.2918, -0.0655, -0.5058],
        [ 0.1068, -0.0103,  0.4649,  ..., -0.1191,  0.1005, -0.3194],
        [ 0.4846,  0.0939, -0.4680,  ..., -0.0723,  0.4256,  0.2847],
        ...,
        [ 0.1377, -0.0487,  0.1247,  ..., -0.3462,  0.5105,  0.0934],
        [-0.1571,  0.3470, -0.1651,  ...,  0.4462, -0.0574,  0.0732],
        [-0.6304, -0.1252,  0.3190,  ..., -0.0235,  0.5926, -0.4541]],
       grad_fn=<ViewBackward0>)